In [ ]:
# ── Step 1: Clean up old processes ─────────────────────────────────────────
import subprocess
subprocess.run(['pkill', '-f', 'cloudflared'], capture_output=True)
subprocess.run(['pkill', '-f', 'main.py'], capture_output=True)
print('✅ Cleaned up old processes')

# ── Step 2: Install ComfyUI ────────────────────────────────────────────────
import os
if not os.path.exists('/kaggle/working/ComfyUI'):
    print('Installing ComfyUI...')
    !git clone -q https://github.com/comfyanonymous/ComfyUI.git /kaggle/working/ComfyUI
    %cd /kaggle/working/ComfyUI
    !pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu121
else:
    print('✅ ComfyUI already installed')
    %cd /kaggle/working/ComfyUI

# ── Step 3: Auto-Link SDXL Model & Download necessary small models ─────────
import os
print('Finding and Linking SDXL model from Dataset...')
os.makedirs('models/checkpoints', exist_ok=True)
os.makedirs('models/upscale_models', exist_ok=True)
os.makedirs('models/vae', exist_ok=True)

model_linked = False
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        if file.endswith('.safetensors') and not file.endswith('vae.safetensors'):
            src_path = os.path.join(root, file)
            if os.path.getsize(src_path) > 2 * 1024 * 1024 * 1024:
                dst_path = 'models/checkpoints/sd_xl_base_1.0.safetensors'
                if os.path.exists(dst_path) or os.path.islink(dst_path):
                    try: os.unlink(dst_path)
                    except: pass
                os.symlink(src_path, dst_path)
                print(f'✅ SDXL Model Linked: {file} -> {dst_path}')
                model_linked = True
                break
    if model_linked:
        break

if not model_linked:
    base_model_path = 'models/checkpoints/sd_xl_base_1.0.safetensors'
    if not os.path.exists(base_model_path):
        print('No SDXL model found in datasets. Downloading base model (this will take 5+ mins)...')
        url = 'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors'
        !wget -q -O {base_model_path} {url}
        print('✅ Base model downloaded successfully.')
    else:
        print('✅ Base model already exists.')

vae_path = 'models/vae/sdxl_vae.safetensors'
if not os.path.exists(vae_path):
    print('Downloading SDXL VAE...')
    url = 'https://huggingface.co/madebyollin/sdxl-vae-fp16-fix/resolve/main/sdxl_vae.safetensors'
    !wget -q -O {vae_path} {url}
    print('✅ VAE downloaded successfully.')
else:
    print('✅ VAE already exists.')

upscaler_path = 'models/upscale_models/RealESRGAN_x4plus.pth'
if not os.path.exists(upscaler_path):
    print('Downloading Real-ESRGAN x4plus...')
    url = 'https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth'
    !wget -q -O {upscaler_path} {url}
    print('✅ Upscaler model downloaded successfully.')
else:
    print('✅ Upscaler model already exists.')

# ── Step 4: Install Cloudflared
if not os.path.exists('/kaggle/working/cloudflared'):
    !wget -q -c https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
    !chmod +x /kaggle/working/cloudflared
print('✅ Cloudflared ready')

# ── Step 5: Cloudflare tunnel + URL reporter 
import re, threading, time, socket
from IPython.display import display, HTML

def iframe_thread(port):
    for _ in range(120):
        time.sleep(1)
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        if sock.connect_ex(('127.0.0.1', port)) == 0:
            sock.close()
            break
        sock.close()
    print('\nComfyUI is running. Starting Cloudflare tunnel...')

    p = subprocess.Popen(
        ['/kaggle/working/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in p.stdout:
        l = line.decode(errors='ignore')
        m = re.search(r'https?://[a-zA-Z0-9-]+\.trycloudflare\.com', l)
        if m:
            url = m.group(0)
            print('\n=======================================================')
            print(f'\u2705 YOUR SERVER URL: {url}')
            display(HTML(f'<a href="{url}" target="_blank" style="font-size:1.2em;font-weight:bold">{url}</a>'))
            print('=======================================================\n')
            break

threading.Thread(target=iframe_thread, daemon=True, args=(8188,)).start()

# ── Step 6: Launch ComfyUI
print('\n🚀 Starting ComfyUI...')
!python main.py --dont-print-server --enable-cors-header "*"